In [60]:
#!/usr/bin/env python
# coding: utf-8

import os
import subprocess
import concurrent.futures
from joblib import Parallel, delayed

In [69]:
# Fetch all existing instance names matching tsm-sc-*
fetch_cmd = f'''
gcloud compute instances list \
    --filter="name~'tsm-sc-'" \
    --format="value(name)"
'''
instances_to_delete = subprocess.check_output(fetch_cmd, shell=True).decode().strip().split('\n')
instances_to_delete = [name for name in instances_to_delete if name]  # Remove empty entries

print("\n➡ Existing instances to delete:", instances_to_delete)

if instances_to_delete:
    # Use parallel deletion
    def delete_instance(instance_name):
        cmd = f'''
        gcloud compute instances delete {instance_name} \
            --zone={zone} \
            --project={project} \
            --quiet
        '''
        print(f"Deleting: {instance_name}")
        return subprocess.call(cmd, shell=True)

    with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
        futures = [executor.submit(delete_instance, inst) for inst in instances_to_delete]
        concurrent.futures.wait(futures)

    print("🧹 All old tsm-sc-* instances deleted.\n")
else:
    print("✔ No previous instances found to delete.\n")


➡ Existing instances to delete: ['tsm-sc-000', 'tsm-sc-001', 'tsm-sc-002', 'tsm-sc-003']
Deleting: tsm-sc-000
Deleting: tsm-sc-001
Deleting: tsm-sc-002
Deleting: tsm-sc-003


Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-000].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-003].
Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-001].


🧹 All old tsm-sc-* instances deleted.



Deleted [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-002].


In [62]:
num_nodes = 4
project = "ucr-ursa-major-lesani-lab"
zone = "us-central1-c"
machine_type = "e2-highmem-2"
image_family = "tsm-sc-family"  # your custom image
subnet = "default"
gcp_username = "tejas"

# Cleanup any existing instances with same prefix
os.system(f'gcloud compute instances delete --zone={zone} --quiet '
          f'$(gcloud compute instances list --filter="name~\'tsm-sc-\'" --format="value(name)")')

# Create commands list
commands = []

for i in range(num_nodes):
    cmd = f'''
    gcloud compute instances create tsm-sc-{i:03} \
        --project={project} \
        --zone={zone} \
        --machine-type={machine_type} \
        --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet={subnet} \
        --can-ip-forward \
        --maintenance-policy=MIGRATE \
        --provisioning-model=STANDARD \
        --service-account=961693926925-compute@developer.gserviceaccount.com \
        --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append \
        --tags=http-server,https-server \
        --create-disk=auto-delete=yes,boot=yes,image-family={image_family},mode=rw,size=20,type=pd-balanced \
        --no-shielded-secure-boot \
        --shielded-vtpm \
        --shielded-integrity-monitoring \
        --labels=goog-ec-src=vm_add-gcloud \
        --reservation-affinity=any
    '''
    commands.append(cmd.strip())


def run_command(command):
    print(f"Running: {command}")
    return subprocess.call(command, shell=True)


# #Parallel instance creation

with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    futures = [executor.submit(run_command, cmd) for cmd in commands]
    concurrent.futures.wait(futures)

print("All instances launched.")

# Wait a bit for IPs to propagate
import time
time.sleep(30)

# Get IPs
os.system('gcloud compute instances list --filter="name~\'tsm-sc-\'" '
          '--format="value(networkInterfaces[0].networkIP)" > tsm_ips.txt')

with open('tsm_ips.txt', 'r') as f:
    iplist = [line.strip() for line in f.readlines()]

print("🎯 Instance IPs:", iplist)

ERROR: (gcloud.compute.instances.delete) argument INSTANCE_NAMES [INSTANCE_NAMES ...]: Must be specified.
Usage: gcloud compute instances delete INSTANCE_NAMES [INSTANCE_NAMES ...] [optional flags]
  optional flags may be  --delete-disks | --help | --keep-disks | --zone

For detailed information on this command and its flags, run:
  gcloud compute instances delete --help


Running: gcloud compute instances create tsm-sc-000         --project=ucr-ursa-major-lesani-lab         --zone=us-central1-c         --machine-type=e2-highcpu-32         --network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default         --can-ip-forward         --maintenance-policy=MIGRATE         --provisioning-model=STANDARD         --service-account=961693926925-compute@developer.gserviceaccount.com         --scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append         --tags=http-server,https-server         --create-disk=auto-delete=yes,boot=yes,image=tsm-sc-image,mode=rw,size=20,type=pd-balanced         --no-shielded-secure-boot         --shielded-vtpm         --shielded-integrity-monitoring         --labels=goog-

Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-001].


NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP   STATUS
tsm-sc-001  us-central1-c  e2-highcpu-32               10.128.0.9   34.45.245.19  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-003].


NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP   STATUS
tsm-sc-003  us-central1-c  e2-highcpu-32               10.128.0.8   34.44.251.19  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-002].


NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP   STATUS
tsm-sc-002  us-central1-c  e2-highcpu-32               10.128.0.13  34.135.38.51  RUNNING


Created [https://www.googleapis.com/compute/v1/projects/ucr-ursa-major-lesani-lab/zones/us-central1-c/instances/tsm-sc-000].


NAME        ZONE           MACHINE_TYPE   PREEMPTIBLE  INTERNAL_IP  EXTERNAL_IP     STATUS
tsm-sc-000  us-central1-c  e2-highcpu-32               10.128.0.10  35.232.237.253  RUNNING
All instances launched.
🎯 Instance IPs: ['10.128.0.10', '10.128.0.9', '10.128.0.13', '10.128.0.8']


In [63]:
os.system('git add .; git commit -m "this works for 4 nodes "; git push')


[main cd42f27] this works for 4 nodes
 3 files changed, 407 insertions(+), 625 deletions(-)


To github.com:tejas-shivanand-mane/stellar-core.git
   5689b8c..cd42f27  main -> main


0

In [64]:





def git_pull_stellar(i):
    command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
git pull"'
    print(command)
    output = os.system(command)
    print(output)

# Execute in parallel like your example
results = Parallel(n_jobs=20)(delayed(git_pull_stellar)(i) for i in range(len(iplist)))
print(results)

From https://github.com/tejas-shivanand-mane/stellar-core
   e10985a..cd42f27  main       -> origin/main
From https://github.com/tejas-shivanand-mane/stellar-core
   e10985a..cd42f27  main       -> origin/main


Updating e10985a..cd42f27
Fast-forward
 .ipynb_checkpoints/SetupGCP-checkpoint.ipynb |  573 +++++++++
 .ipynb_checkpoints/post-checkpoint.ipynb     |  153 +++
 GCP_setup.sh                                 |   46 +-
 SetupGCP.ipynb                               | 1784 ++++++++++++++++++++++++++
 gcp_setup_stellar_private.sh                 |  150 +++
 latency.png                                  |  Bin 0 -> 61881 bytes
 post.ipynb                                   |  621 +++++++++
 src/main/Config.cpp                          |    8 +-
 src/overlay/OverlayManagerImpl.cpp           |  462 ++++---
 src/protocol-curr/xdr/Stellar-overlay.x      |    2 +-
 throughput.png                               |  Bin 0 -> 79372 bytes
 tsm_ips.txt                                  |    4 +
 12 files changed, 3589 insertions(+), 214 deletions(-)
 create mode 100644 .ipynb_checkpoints/SetupGCP-checkpoint.ipynb
 create mode 100644 .ipynb_checkpoints/post-checkpoint.ipynb
 create mode 100644 SetupGCP.ipynb


From https://github.com/tejas-shivanand-mane/stellar-core
   e10985a..cd42f27  main       -> origin/main


Updating e10985a..cd42f27
Fast-forward
 .ipynb_checkpoints/SetupGCP-checkpoint.ipynb |  573 +++++++++
 .ipynb_checkpoints/post-checkpoint.ipynb     |  153 +++
 GCP_setup.sh                                 |   46 +-
 SetupGCP.ipynb                               | 1784 ++++++++++++++++++++++++++
 gcp_setup_stellar_private.sh                 |  150 +++
 latency.png                                  |  Bin 0 -> 61881 bytes
 post.ipynb                                   |  621 +++++++++
 src/main/Config.cpp                          |    8 +-
 src/overlay/OverlayManagerImpl.cpp           |  462 ++++---
 src/protocol-curr/xdr/Stellar-overlay.x      |    2 +-
 throughput.png                               |  Bin 0 -> 79372 bytes
 tsm_ips.txt                                  |    4 +
 12 files changed, 3589 insertions(+), 214 deletions(-)
 create mode 100644 .ipynb_checkpoints/SetupGCP-checkpoint.ipynb
 create mode 100644 .ipynb_checkpoints/post-checkpoint.ipynb
 create mode 100644 SetupGCP.ipynb


From https://github.com/tejas-shivanand-mane/stellar-core
   e10985a..cd42f27  main       -> origin/main


Updating e10985a..cd42f27
Fast-forward
 .ipynb_checkpoints/SetupGCP-checkpoint.ipynb |  573 +++++++++
 .ipynb_checkpoints/post-checkpoint.ipynb     |  153 +++
 GCP_setup.sh                                 |   46 +-
 SetupGCP.ipynb                               | 1784 ++++++++++++++++++++++++++
 gcp_setup_stellar_private.sh                 |  150 +++
 latency.png                                  |  Bin 0 -> 61881 bytes
 post.ipynb                                   |  621 +++++++++
 src/main/Config.cpp                          |    8 +-
 src/overlay/OverlayManagerImpl.cpp           |  462 ++++---
 src/protocol-curr/xdr/Stellar-overlay.x      |    2 +-
 throughput.png                               |  Bin 0 -> 79372 bytes
 tsm_ips.txt                                  |    4 +
 12 files changed, 3589 insertions(+), 214 deletions(-)
 create mode 100644 .ipynb_checkpoints/SetupGCP-checkpoint.ipynb
 create mode 100644 .ipynb_checkpoints/post-checkpoint.ipynb
 create mode 100644 SetupGCP.ipynb


In [65]:
import shutil

if os.path.exists('../stellar-private'):
    
    shutil.rmtree('../stellar-private')
os.mkdir('../stellar-private')


os.system('cp gcp_setup_stellar_private.sh ../stellar-private/gcp_setup_stellar_private.sh')

os.system('cd ../stellar-private; chmod +x gcp_setup_stellar_private.sh; ./gcp_setup_stellar_private.sh start; ./gcp_setup_stellar_private.sh')

Detected 4 nodes based on tsm_ips.txt.
Cleaning and creating directory for node1. Ports: Peer 11625, HTTP 11626...
Cleaning and creating directory for node2. Ports: Peer 11635, HTTP 11636...
Cleaning and creating directory for node3. Ports: Peer 11645, HTTP 11646...
Cleaning and creating directory for node4. Ports: Peer 11655, HTTP 11656...
Generating seed for node1...
Generating seed for node2...
Generating seed for node3...
Generating seed for node4...
Creating config file for node1...
Creating config file for node2...
Creating config file for node3...
Creating config file for node4...
Detected 4 nodes based on tsm_ips.txt.
Initializing database for node1...
Initializing database for node2...


2025-12-18T11:15:10.238 [default INFO] Config from /home/tejas/stellar-private/node1/stellar-core.cfg
2025-12-18T11:15:10.238 [default INFO] Generated QUORUM_SET: {
   "t" : 3,
   "v" : [ "node2", "node3", "node4", "GDCK4" ]
}

2025-12-18T11:15:10.238 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2025-12-18T11:15:10.238 [default INFO] Assigning calculated value of 1 to FAILURE_SAFETY
2025-12-18T11:15:10.276 [default INFO] Config from /home/tejas/stellar-private/node2/stellar-core.cfg
2025-12-18T11:15:10.277 [default INFO] Generated QUORUM_SET: {
   "t" : 3,
   "v" : [ "GBF5P", "node3", "node4", "node1" ]
}

2025-12-18T11:15:10.277 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2025-12-18T11:15:10.277 [default INFO] Assigning calculated value of 1 to FAILURE_SAFETY
2025-12-18T11:15:10.338 [default INFO] Config from /home/tejas/stellar-private/node3/ste

Initializing database for node3...
Initializing database for node4...
✅ 4-node private Stellar network setup complete!
Start the nodes with (substituting node number):
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node1/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node2/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node3/stellar-core.cfg &
/home/tejas/stellar-core/src/stellar-core run --conf /home/tejas/stellar-private/node4/stellar-core.cfg &


2025-12-18T11:15:10.376 [default INFO] Config from /home/tejas/stellar-private/node4/stellar-core.cfg
2025-12-18T11:15:10.377 [default INFO] Generated QUORUM_SET: {
   "t" : 3,
   "v" : [ "node2", "node3", "GBNHK", "node1" ]
}

2025-12-18T11:15:10.377 [default WARNING] Adjusted TARGET_PEER_CONNECTIONS to 336 due to insufficient MAX_ADDITIONAL_PEER_CONNECTIONS=1000
2025-12-18T11:15:10.377 [default INFO] Assigning calculated value of 1 to FAILURE_SAFETY


0

In [67]:
# --- Configuration ---
line_to_add = "SEND_CUSTOM_MESSAGE=true"
target_file = "../stellar-private/node1/stellar-core.cfg" 

# --- The os.system() Command ---
# This command prepends the line to the target_file on your local machine.
os.system(f'(echo "{line_to_add}"; cat {target_file}) > {target_file}.tmp && mv {target_file}.tmp {target_file}')

print(f"The line '{line_to_add}' has been prepended to {target_file}.")

The line 'SEND_CUSTOM_MESSAGE=true' has been prepended to ../stellar-private/node1/stellar-core.cfg.


In [68]:
def compile_stellar(i):
    command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
cd stellar-core; \
make -j16; cd; sudo rm -r stellar-private"'
    print(command)
    output = os.system(command)
    print(output)

# Execute in parallel like your example
results = Parallel(n_jobs=20)(delayed(compile_stellar)(i) for i in range(len(iplist)))
print(results)

make  all-recursive
make  all-recursive
make[1]: Entering directory '/home/tejas/stellar-core'
make[1]: Entering directory '/home/tejas/stellar-core'
Making all in lib
Making all in lib
make  all-recursive
make[1]: Entering directory '/home/tejas/stellar-core'
Making all in lib
make[2]: Entering directory '/home/tejas/stellar-core/lib'
Making all in ../lib/libsodium
make[2]: Entering directory '/home/tejas/stellar-core/lib'
make[3]: Entering directory '/home/tejas/stellar-core/lib/libsodium'
Making all in ../lib/libsodium
Making all in builds
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/builds'
make[4]: Nothing to be done for 'all'.
make[4]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/builds'
Making all in contrib
make[3]: Entering directory '/home/tejas/stellar-core/lib/libsodium'
make[4]: Entering directory '/home/tejas/stellar-core/lib/libsodium/contrib'
make[4]: Nothing to be done for 'all'.
make[4]: Leaving directory '/home/tejas/stellar-core/l

/bin/bash: line 1: pandoc: command not found
make[4]: [Makefile:1923: doc/xdrc.1] Error 127 (ignored)
/bin/bash: line 1: pandoc: command not found
make[4]: [Makefile:1923: doc/xdrc.1] Error 127 (ignored)
/bin/bash: line 1: pandoc: command not found
make[4]: [Makefile:1923: doc/xdrc.1] Error 127 (ignored)


make[5]: Entering directory '/home/tejas/stellar-core/lib/libsodium/src/libsodium'
Making all in include
make[6]: Entering directory '/home/tejas/stellar-core/lib/libsodium/src/libsodium/include'
make[6]: Nothing to be done for 'all'.
make[6]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/src/libsodium/include'
make[3]: Entering directory '/home/tejas/stellar-core/lib'
make[3]: Nothing to be done for 'all-am'.
make[3]: Leaving directory '/home/tejas/stellar-core/lib'
make[2]: Leaving directory '/home/tejas/stellar-core/lib'
Making all in src
make[5]: Entering directory '/home/tejas/stellar-core/lib/libsodium/test/default'
make[5]: Nothing to be done for 'all'.
make[5]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/test/default'
make[5]: Entering directory '/home/tejas/stellar-core/lib/libsodium/test'
make[5]: Nothing to be done for 'all-am'.
make[5]: Leaving directory '/home/tejas/stellar-core/lib/libsodium/test'
make[4]: Leaving directory '/home/tejas/stellar-c

/bin/bash: line 1: pandoc: command not found
make[4]: [Makefile:1923: doc/xdrc.1] Error 127 (ignored)


make[3]: Entering directory '/home/tejas/stellar-core/lib'
make[3]: Nothing to be done for 'all-am'.
make[3]: Leaving directory '/home/tejas/stellar-core/lib'
make[2]: Leaving directory '/home/tejas/stellar-core/lib'
Making all in src
make[2]: Entering directory '/home/tejas/stellar-core/src'
../lib/xdrpp/xdrc/xdrc -hh -pedantic -o protocol-curr/xdr/Stellar-overlay.h protocol-curr/xdr/Stellar-overlay.x
../hash-xdrs.sh protocol-curr >main/XDRFilesSha256.cpp
make[2]: Entering directory '/home/tejas/stellar-core/src'
../lib/xdrpp/xdrc/xdrc -hh -pedantic -o protocol-curr/xdr/Stellar-overlay.h protocol-curr/xdr/Stellar-overlay.x
../hash-xdrs.sh protocol-curr >main/XDRFilesSha256.cpp
echo '#include "main/StellarCoreVersion.h"

const std::string STELLAR_CORE_VERSION = "cd42f27";' > main/StellarCoreVersion.cpp
make[2]: Entering directory '/home/tejas/stellar-core/src'
../lib/xdrpp/xdrc/xdrc -hh -pedantic -o protocol-curr/xdr/Stellar-overlay.h protocol-curr/xdr/Stellar-overlay.x
../hash-xdrs.sh

...

Exception ignored in: <function ResourceTracker.__del__ at 0x79e65878e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


gcloud compute ssh --zone "us-central1-c" "tsm-sc-000" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; make -j16; cd; sudo rm -r stellar-private"
256


Exception ignored in: <function ResourceTracker.__del__ at 0x78bdaf78e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


gcloud compute ssh --zone "us-central1-c" "tsm-sc-001" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; make -j16; cd; sudo rm -r stellar-private"
256


Exception ignored in: <function ResourceTracker.__del__ at 0x7111b7d7e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


In [51]:
def clean_stellar_private(i):
    remote_command = f"""\
cd /home/tejas; \
sudo rm -r stellar-private; \
"""
    
    # Construct the full gcloud command
    command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "{remote_command}"'
    
    print(f"Executing: {command}")

    output = os.system(command)
    print(f"Return code for tsm-sc-{i:03}: {output}")


results = Parallel(n_jobs=20)(delayed(clean_stellar_private)(i) for i in range(num_nodes))

rm: cannot remove 'stellar-private': No such file or directory
rm: cannot remove 'stellar-private': No such file or directory
rm: cannot remove 'stellar-private': No such file or directory
rm: cannot remove 'stellar-private': No such file or directory


In [52]:
def copy_folder_to_instance(i, source_folder = '/home/tejas/stellar-private',destination_path = '/home/tejas/stellar-private' ):
    """
    Constructs and executes the gcloud compute scp command to copy a folder
    to a specific GCP instance.
    """
    instance_name = f"tsm-sc-{i:03}"
    
    # The --recurse flag is crucial for copying folders
    # The format is: gcloud compute scp --recurse [LOCAL_SRC] [USER]@[INSTANCE_NAME]:[REMOTE_DEST]
    command = f'gcloud compute scp --zone "{zone}" --project "{project}" \
--recurse "{source_folder}" "{instance_name}:{destination_path}"'

    print(f"Executing command for {instance_name}: {command}")
    
    # os.system executes the command and returns the exit status (0 for success)
    output = os.system(command)
    
    print(f"Command for {instance_name} finished with exit code: {output}")
    
    return (instance_name, output)


results = Parallel(n_jobs=20)(
    delayed(copy_folder_to_instance)(i) for i in range(num_nodes)
)

In [53]:
# def setup_stellar_private(i):
#     command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "\
# cd /home/tejas; \
# cd stellar-private; chmod +x gcp_setup_stellar_private.sh; \
# ./gcp_setup_stellar_private.sh;"'
#     print(command)
#     output = os.system(command)
#     print(output)

# # Execute in parallel like your example
# results = Parallel(n_jobs=20)(delayed(setup_stellar_private)(i) for i in range(len(iplist)))
# print(results)

In [54]:
def run_stellar_private(i):
    # Calculate the node number (assuming i starts at 0, node starts at 1)
    node_number = i + 1 
    instance_name = f"tsm-sc-{i:03}"
    
    # ----------------------------------------------------------------------------------
    # FIX: Use nohup, redirect I/O to a log file, and add ' & disown'
    # '2>&1' redirects stderr to stdout. '> log.txt' redirects stdout to a file.
    # '< /dev/null' ensures the process doesn't wait for input.
    # ----------------------------------------------------------------------------------
    remote_command = f"""\
cd /home/tejas/stellar-private; \
nohup /home/tejas/stellar-core/src/stellar-core run --conf node{node_number}/stellar-core.cfg \
> node{node_number}/stellar-core.log 2>&1 < /dev/null & disown
"""
    
    # Construct the full gcloud command
    command = f'gcloud compute ssh --zone "{zone}" "{instance_name}" --project "{project}" --command "{remote_command}"'
    
    print(f"Executing: {command}")
    
    # os.system should now return immediately because the remote shell exits
    output = os.system(command)
    print(f"Return code for {instance_name}: {output}")

# Corrected Loop (to run 0, 1, 2, 3)
results = Parallel(n_jobs=20)(delayed(run_stellar_private)(i) for i in range(num_nodes))
# results = Parallel(n_jobs=20)(delayed(run_stellar_private)(i) for i in [3,2,1,0])

# time.sleep(3)
# for i in range(num_nodes):

    # run_stellar_private(num_nodes-i-1)
    # time.sleep(2)
# run_stellar_private(0)
print(results)
print("All SSH commands executed. Nodes should be starting up in the background.")

[None, None, None, None]
All SSH commands executed. Nodes should be starting up in the background.


In [55]:
time.sleep(60)

gcloud compute ssh --zone "us-central1-c" "tsm-sc-001" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; git pull"
0
gcloud compute ssh --zone "us-central1-c" "tsm-sc-000" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; git pull"
0


Exception ignored in: <function ResourceTracker.__del__ at 0x739117b8a020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x744e99186020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


gcloud compute ssh --zone "us-central1-c" "tsm-sc-002" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; git pull"
0
gcloud compute ssh --zone "us-central1-c" "tsm-sc-003" --project "ucr-ursa-major-lesani-lab" --command "cd stellar-core; git pull"
0


Exception ignored in: <function ResourceTracker.__del__ at 0x7b7402d92020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x75ce53386020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


In [56]:


def kill_stellar_private(i):
    remote_command = f"""\
cd /home/tejas/stellar-private; \
sudo pkill stellar-core; \
"""
    
    # Construct the full gcloud command
    command = f'gcloud compute ssh --zone "{zone}" "tsm-sc-{i:03}" --project "{project}" --command "{remote_command}"'
    
    print(f"Executing: {command}")

    output = os.system(command)
    print(f"Return code for tsm-sc-{i:03}: {output}")


results = Parallel(n_jobs=20)(delayed(kill_stellar_private)(i) for i in range(num_nodes))


In [57]:
remote_base_folder = "/home/tejas/stellar-private" # The base path on the GCP instance
local_base_destination = "/home/tejas/work/experiments/stellar-core/" + "collection_" + str(num_nodes) 

# Ensure the local base destination directory exists
os.makedirs(local_base_destination, exist_ok=True)


def copy_folder_from_instance(i):
    """
    Constructs and executes the gcloud compute scp command to copy a specific 
    nodeN folder from instance i to a local folder named after the instance.
    """
    instance_name = f"tsm-sc-{i:03}"
    
    # Calculate the node number (assuming i starts at 0, node starts at 1)
    node_number = i + 1 
    node_folder = f"node{node_number}"

    # 1. Define the specific REMOTE source path on the instance
    # Example: /home/tejas/stellar-private/node1
    remote_source_path = os.path.join(remote_base_folder, node_folder)
    
    # 2. Define the LOCAL destination path
    # We'll use the instance name for the subfolder to keep backups separate
    local_destination_path = os.path.join(local_base_destination, instance_name)
    os.makedirs(local_destination_path, exist_ok=True)
    
    # The SCp command requires the remote path to be formatted as:
    # [INSTANCE_NAME]:[REMOTE_SRC]
    remote_source = f"{instance_name}:{remote_source_path}"
    
    # The command reverses the source (remote) and destination (local)
    command = f'gcloud compute scp --zone "{zone}" --project "{project}" \
--recurse "{remote_source}" "{local_destination_path}"'

    print(f"Executing command to copy {node_folder} from {instance_name}: {command}")
    
    # os.system executes the command and returns the exit status (0 for success)
    output = os.system(command)
    
    print(f"Copy from {instance_name} finished with exit code: {output}")
    
    return (instance_name, output)

# ---
# Execute the copy operation in parallel
# ---

results = Parallel(n_jobs=20)(
    delayed(copy_folder_from_instance)(i) for i in range(num_nodes)
)

print("\n--- Summary of Download Results ---")
print(results)


--- Summary of Download Results ---
[('tsm-sc-000', 0), ('tsm-sc-001', 0), ('tsm-sc-002', 0), ('tsm-sc-003', 0)]


In [58]:

# # Fetch all existing instance names matching tsm-sc-*
# fetch_cmd = f'''
# gcloud compute instances list \
#     --filter="name~'tsm-sc-'" \
#     --format="value(name)"
# '''
# instances_to_delete = subprocess.check_output(fetch_cmd, shell=True).decode().strip().split('\n')
# instances_to_delete = [name for name in instances_to_delete if name]  # Remove empty entries

# print("\n➡ Existing instances to delete:", instances_to_delete)

# if instances_to_delete:
#     # Use parallel deletion
#     def delete_instance(instance_name):
#         cmd = f'''
#         gcloud compute instances delete {instance_name} \
#             --zone={zone} \
#             --project={project} \
#             --quiet
#         '''
#         print(f"Deleting: {instance_name}")
#         return subprocess.call(cmd, shell=True)

#     with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
#         futures = [executor.submit(delete_instance, inst) for inst in instances_to_delete]
#         concurrent.futures.wait(futures)

#     print("🧹 All old tsm-sc-* instances deleted.\n")
# else:
#     print("✔ No previous instances found to delete.\n")


In [59]:
# # Fetch all existing instance names matching tsm-sc-*
# fetch_cmd = f'''
# gcloud compute instances list \
#     --filter="name~'tsm-sc-'" \
#     --format="value(name)"
# '''
# instances_to_delete = subprocess.check_output(fetch_cmd, shell=True).decode().strip().split('\n')
# instances_to_delete = [name for name in instances_to_delete if name]  # Remove empty entries

# print("\n➡ Existing instances to delete:", instances_to_delete)

# if instances_to_delete:
#     # Use parallel deletion
#     def delete_instance(instance_name):
#         cmd = f'''
#         gcloud compute instances delete {instance_name} \
#             --zone={zone} \
#             --project={project} \
#             --quiet
#         '''
#         print(f"Deleting: {instance_name}")
#         return subprocess.call(cmd, shell=True)

#     with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
#         futures = [executor.submit(delete_instance, inst) for inst in instances_to_delete]
#         concurrent.futures.wait(futures)

#     print("🧹 All old tsm-sc-* instances deleted.\n")
# else:
#     print("✔ No previous instances found to delete.\n")

Executing command for tsm-sc-001: gcloud compute scp --zone "us-central1-c" --project "ucr-ursa-major-lesani-lab" --recurse "/home/tejas/stellar-private" "tsm-sc-001:/home/tejas/stellar-private"
Command for tsm-sc-001 finished with exit code: 0
Executing command for tsm-sc-002: gcloud compute scp --zone "us-central1-c" --project "ucr-ursa-major-lesani-lab" --recurse "/home/tejas/stellar-private" "tsm-sc-002:/home/tejas/stellar-private"
Command for tsm-sc-002 finished with exit code: 0
Executing command for tsm-sc-000: gcloud compute scp --zone "us-central1-c" --project "ucr-ursa-major-lesani-lab" --recurse "/home/tejas/stellar-private" "tsm-sc-000:/home/tejas/stellar-private"
Command for tsm-sc-000 finished with exit code: 0


Exception ignored in: <function ResourceTracker.__del__ at 0x7f980558a020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7ec27257a020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ 

Executing command for tsm-sc-003: gcloud compute scp --zone "us-central1-c" --project "ucr-ursa-major-lesani-lab" --recurse "/home/tejas/stellar-private" "tsm-sc-003:/home/tejas/stellar-private"
Command for tsm-sc-003 finished with exit code: 0


Exception ignored in: <function ResourceTracker.__del__ at 0x7d4ca1d8e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-003" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas/stellar-private; nohup /home/tejas/stellar-core/src/stellar-core run --conf node4/stellar-core.cfg > node4/stellar-core.log 2>&1 < /dev/null & disown
"
Return code for tsm-sc-003: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-000" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas/stellar-private; nohup /home/tejas/stellar-core/src/stellar-core run --conf node1/stellar-core.cfg > node1/stellar-core.log 2>&1 < /dev/null & disown
"
Return code for tsm-sc-000: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-002" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas/stellar-private; nohup /home/tejas/stellar-core/src/stellar-core run --conf node3/stellar-core.cfg > node3/stellar-core.log 2>&1 < /dev/null & disown
"
Return code for tsm-sc-002: 0
Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-001

...

Exception ignored in: <function ResourceTracker.__del__ at 0x784024b92020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7329a9d8e020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-001" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas; sudo rm -r stellar-private; "
Return code for tsm-sc-001: 256
Executing command to copy node4 from tsm-sc-003: gcloud compute scp --zone "us-central1-c" --project "ucr-ursa-major-lesani-lab" --recurse "tsm-sc-003:/home/tejas/stellar-private/node4" "/home/tejas/work/experiments/stellar-core/collection_4/tsm-sc-003"
Copy from tsm-sc-003 finished with exit code: 0


Exception ignored in: <function ResourceTracker.__del__ at 0x785fcf986020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


Executing: gcloud compute ssh --zone "us-central1-c" "tsm-sc-003" --project "ucr-ursa-major-lesani-lab" --command "cd /home/tejas; sudo rm -r stellar-private; "
Return code for tsm-sc-003: 256
Executing command to copy node1 from tsm-sc-000: gcloud compute scp --zone "us-central1-c" --project "ucr-ursa-major-lesani-lab" --recurse "tsm-sc-000:/home/tejas/stellar-private/node1" "/home/tejas/work/experiments/stellar-core/collection_4/tsm-sc-000"
Copy from tsm-sc-000 finished with exit code: 0


Exception ignored in: <function ResourceTracker.__del__ at 0x7a55ebd82020>
Traceback (most recent call last):
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/tejas/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
